# Assignment 3: LLM-Powered Applications and Distributed Computing

**NAME:** Varsha Roopchand  
**ID:** 816039243  
**Course:** COMP 3610 Big Data Analytics  


## Notebook Setup

This first section installs the packages required by the assignment, defines project paths, and configures the LLM client used later for the RAG pipeline, query router, and natural-language-to-SQL components.

In [ ]:
# Note: This notebook was run on google colab so this cell block was run to match the compatibility, but is unneccesary if something else is used to run the code
%pip install -q \
  requests==2.32.5 \
  opentelemetry-api==1.40.0 \
  opentelemetry-sdk==1.40.0 \
  opentelemetry-proto==1.40.0 \
  opentelemetry-exporter-otlp-proto-common==1.40.0 \
  opentelemetry-exporter-otlp-proto-http==1.40.0


In [ ]:
# Note: This was run on google colab so "!" would have to switch to "%" if run elsewhere
!pip install -q pandas numpy requests matplotlib pypdf openai pyspark chromadb sentence-transformers langchain langchain-community langchain-text-splitters duckdb

In [ ]:
import json
import os, subprocess
import time
import re
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from IPython.display import Markdown, display
from openai import OpenAI
from pypdf import PdfReader

from pyspark.sql import SparkSession, functions as F


In [ ]:
# Note: This was run on google colab which had a compatible version of java. If run elsewhere, the compatible version may have to be set manually.
# e.g
#os.environ["PYSPARK_PYTHON"] = sys.executable
#os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Force local networking on Windows
#os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"
#os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot"
#os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

java_version_output = subprocess.check_output(["java", "-version"], stderr=subprocess.STDOUT, text=True)
print("Java Version: ",java_version_output)
match = re.search(r'version \"(\\d+)', java_version_output)
if match:
    major = int(match.group(1))
    if major not in {8, 11, 17}:
        raise RuntimeError("Unsupported Java version for this notebook. Install Java 17, set JAVA_HOME to it, restart the kernel, and run again.")

RANDOM_STATE = 42

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
ARTIFACTS_DIR = ROOT / "artifacts"
CHROMA_ROOT = ARTIFACTS_DIR / "chroma"
DOCS_DIR = ROOT / "docs"

for path in [RAW_DIR, PROCESSED_DIR, ARTIFACTS_DIR, CHROMA_ROOT, DOCS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

TRIP_DATA_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
ZONE_LOOKUP_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

TRIP_PARQUET_PATH = RAW_DIR / "yellow_tripdata_2024-01.parquet"
ZONE_LOOKUP_PATH = RAW_DIR / "taxi_zone_lookup.csv"
CLEANED_SPARK_PATH = PROCESSED_DIR / "yellow_taxi_clean_spark"

# Note: Add API KEY
LLM_BASE_URL = os.getenv("LLM_BASE_URL", "https://synapse.sergiomathurin.com/v1")
LLM_API_KEY = os.getenv("SYNAPSE_API_KEY", "<INSERT KEY HERE>")
LLM_MODEL = os.getenv("LLM_MODEL", "llama3.3-70b-instruct")

print(f"Project root: {ROOT.resolve()}")
print(f"Docs directory: {DOCS_DIR.resolve()}")
print(f"LLM model: {LLM_MODEL}")
print("API key loaded from environment." if LLM_API_KEY else "No API key found yet; LLM sections will remain inactive until SYNAPSE_API_KEY is set.")


def make_llm_client() -> OpenAI:
    if not LLM_API_KEY:
        raise ValueError("Set SYNAPSE_API_KEY in your environment before running LLM-backed sections.")
    return OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)


def timed_call(fn, *args, **kwargs):
    start = time.perf_counter()
    result = fn(*args, **kwargs)
    elapsed = time.perf_counter() - start
    return result, elapsed


def download_file(url: str, destination: Path) -> Path:
    if destination.exists():
        print(f"Using cached file: {destination.name}")
        return destination

    response = requests.get(url, stream=True, timeout=120)
    response.raise_for_status()

    with destination.open("wb") as handle:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                handle.write(chunk)

    print(f"Downloaded: {destination.name}")
    return destination

## Part 1: Distributed Data Processing with Spark



### Task 1.1: Spark Environment Setup and Data Loading



In [ ]:
# Download the Data
download_file(TRIP_DATA_URL, TRIP_PARQUET_PATH)
download_file(ZONE_LOOKUP_URL, ZONE_LOOKUP_PATH)

# Create a SparkSession with appropriate configuration
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("COMP3610_Assignment3")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f'Spark version: {spark.version}')
print(f'App name: {spark.sparkContext.appName}')
print(f'Master: {spark.sparkContext.master}')
print(f'Default parallelism: {spark.sparkContext.defaultParallelism}')
print('Spark UI: http://localhost:4040')

for key, value in sorted(spark.sparkContext.getConf().getAll()):
    if 'memory' in key.lower() or 'core' in key.lower() or 'master' in key.lower():
        print(f'{key} = {value}')

# Load the NYC Yellow Taxi Parquet data into a Spark DataFrame
spark_df, spark_load_time = timed_call(spark.read.parquet, str(TRIP_PARQUET_PATH))
pandas_df, pandas_load_time = timed_call(pd.read_parquet, TRIP_PARQUET_PATH)
zones_pd = pd.read_csv(ZONE_LOOKUP_PATH)
zones_spark = spark.createDataFrame(zones_pd)

#Report the total row count and DataFrame partition count and compare load time against Pandas for the same file
print(f"Spark load time : {spark_load_time:.2f} seconds")
print(f"Pandas load time: {pandas_load_time:.2f} seconds")
print(f"Spark row count : {spark_df.count():,}")
print(f"Spark partitions: {spark_df.rdd.getNumPartitions()}")
print(f"Pandas shape    : {pandas_df.shape}")

# Display the schema and verify column type
spark_df.printSchema()

display(
    pd.DataFrame(
        {
            "engine": ["Spark", "Pandas"],
            "load_seconds": [spark_load_time, pandas_load_time],
        }
    )
)


The Spark environment was configured successfully in local mode with local[*], a driver memory setting of 4g, and Spark version 4.0.2. The January 2024 NYC Yellow Taxi dataset loaded correctly, with both Spark and Pandas confirming 2,964,624 records, while the Spark schema shows that the key timestamp, location, and fare-related columns were assigned appropriate data types for downstream analysis.

For the load time comparison, Pandas loaded the single Parquet file faster (2.14 seconds) than Spark (3.30 seconds). This is expected in a local Colab environment because Spark has extra session startup and distributed-processing overhead, whereas Pandas reads a single file directly into memory with less setup cost. Even though Pandas was faster for this one-file load, Spark is still more suitable for the rest of the assignment because it is designed for scalable transformations, SQL analytics, and repeated processing on large datasets.

### Task 1.2: Data Cleaning and Feature Engineering in Spark

In [ ]:
CRITICAL_NULL_COLS = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "fare_amount",
    "trip_distance",
]

# Remove rows with null values in critical columns (pickup/dropoff times, locations, fare, distance) and Filter out invalid trips: zero/negative distance, negative fares, fares exceeding $500, dropoff before pickup
def clean_and_engineer_taxi_data(df):
    counts = {}
    counts["starting_rows"] = df.count()

    step1 = df.dropna(subset=CRITICAL_NULL_COLS)
    counts["after_drop_nulls"] = step1.count()
    counts["removed_nulls"] = counts["starting_rows"] - counts["after_drop_nulls"]

    step2 = step1.filter(F.col("trip_distance") > 0)
    counts["after_positive_distance"] = step2.count()
    counts["removed_nonpositive_distance"] = counts["after_drop_nulls"] - counts["after_positive_distance"]

    step3 = step2.filter(F.col("fare_amount") >= 0)
    counts["after_nonnegative_fare"] = step3.count()
    counts["removed_negative_fare"] = counts["after_positive_distance"] - counts["after_nonnegative_fare"]

    step4 = step3.filter(F.col("fare_amount") <= 500)
    counts["after_fare_cap"] = step4.count()
    counts["removed_fare_above_500"] = counts["after_nonnegative_fare"] - counts["after_fare_cap"]

    step5 = step4.filter(F.col("tpep_dropoff_datetime") >= F.col("tpep_pickup_datetime"))
    counts["final_rows"] = step5.count()
    counts["removed_bad_timestamp_order"] = counts["after_fare_cap"] - counts["final_rows"]

    duration_minutes = (
        F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")
    ) / 60.0

# Create derived columns using Spark functions
    cleaned = (
        step5
        .withColumn("trip_duration_minutes", duration_minutes.cast("double"))
        .withColumn(
            "trip_speed_mph",
            F.when(F.col("trip_duration_minutes") > 0,
                   F.col("trip_distance") / (F.col("trip_duration_minutes") / 60.0))
            .otherwise(F.lit(None).cast("double"))
        )
        .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
        .withColumn("pickup_day_of_week", F.dayofweek("tpep_pickup_datetime"))
        .withColumn(
            "tip_percentage",
            F.when(F.col("fare_amount") > 0, (F.col("tip_amount") / F.col("fare_amount")) * 100.0)
            .otherwise(F.lit(None).cast("double"))
        )
    )
    return cleaned, counts

# Print the count of rows removed at each cleaning step
clean_df, cleaning_counts = clean_and_engineer_taxi_data(spark_df)

cleaning_report = pd.DataFrame(
    [{"step": key, "rows": value} for key, value in cleaning_counts.items()]
)
display(cleaning_report)

clean_df.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "fare_amount",
    "trip_duration_minutes",
    "trip_speed_mph",
    "pickup_hour",
    "pickup_day_of_week",
    "tip_percentage",
).show(5, truncate=False)

The Spark-based cleaning process reduced the dataset from 2,964,624 rows to 2,870,102 valid trips, meaning 94,522 records were removed overall. No rows were removed for missing values in the critical columns, which suggests the raw dataset is structurally complete, while most removals came from invalid trip distances (60,371 rows) and negative fares (34,065 rows), showing that the main data quality issues were anomalous trip records rather than missing data.

Only a very small number of rows were excluded for fares above $500 (30 rows) and for invalid timestamp ordering where dropoff occurred before pickup (56 rows), indicating that these issues were rare. The preview of the engineered columns shows that trip_duration_minutes, trip_speed_mph, pickup_hour, pickup_day_of_week, and tip_percentage were created successfully and contain realistic values, confirming that the cleaning and feature engineering steps were implemented correctly for downstream Spark SQL analysis.

### Task 1.3: Spark SQL Analytics



In [ ]:
clean_df.createOrReplaceTempView("taxi_trips_clean")
zones_spark.createOrReplaceTempView("taxi_zones")

queries = {
    # Query 1: What are the top 10 busiest pickup hours, and what is the average fare and tip percentage for each?
    "query_1_busiest_hours": """
        SELECT
            pickup_hour,
            COUNT(*) AS trip_count,
            ROUND(AVG(fare_amount), 2) AS avg_fare,
            ROUND(AVG(tip_percentage), 2) AS avg_tip_percentage
        FROM taxi_trips_clean
        GROUP BY pickup_hour
        ORDER BY trip_count DESC
        LIMIT 10
    """,
    # Query 2: Which day of the week has the highest average trip speed? Include average distance and duration
    "query_2_fastest_day": """
        SELECT
            pickup_day_of_week,
            ROUND(AVG(trip_speed_mph), 2) AS avg_trip_speed_mph,
            ROUND(AVG(trip_distance), 2) AS avg_trip_distance,
            ROUND(AVG(trip_duration_minutes), 2) AS avg_trip_duration_minutes
        FROM taxi_trips_clean
        WHERE trip_speed_mph IS NOT NULL
        GROUP BY pickup_day_of_week
        ORDER BY avg_trip_speed_mph DESC
    """,
    # Query 3: Using a window function, rank the top 5 pickup locations by total revenue for each day of the week
    "query_3_revenue_rank_by_day": """
        WITH revenue_by_location AS (
            SELECT
                pickup_day_of_week,
                PULocationID,
                SUM(total_amount) AS total_revenue
            FROM taxi_trips_clean
            GROUP BY pickup_day_of_week, PULocationID
        )
        SELECT *
        FROM (
            SELECT
                r.pickup_day_of_week,
                r.PULocationID,
                z.Zone AS pickup_zone,
                ROUND(r.total_revenue, 2) AS total_revenue,
                ROW_NUMBER() OVER (
                    PARTITION BY r.pickup_day_of_week
                    ORDER BY r.total_revenue DESC
                ) AS revenue_rank
            FROM revenue_by_location r
            LEFT JOIN taxi_zones z
              ON r.PULocationID = z.LocationID
        ) ranked
        WHERE revenue_rank <= 5
        ORDER BY pickup_day_of_week, revenue_rank
    """,
    # Query 4: Calculate the cumulative trip count by hour of day (running total from hour 0 to 23). At what hour does the cumulative count surpass 50% of daily trips?
    "query_4_cumulative_hourly_trips": """
        WITH hourly_counts AS (
            SELECT pickup_hour, COUNT(*) AS trip_count
            FROM taxi_trips_clean
            GROUP BY pickup_hour
        ),
        cumulative AS (
            SELECT
                pickup_hour,
                trip_count,
                SUM(trip_count) OVER (ORDER BY pickup_hour) AS cumulative_trip_count,
                SUM(trip_count) OVER () AS total_trip_count
            FROM hourly_counts
        )
        SELECT
            pickup_hour,
            trip_count,
            cumulative_trip_count,
            ROUND(cumulative_trip_count / total_trip_count * 100, 2) AS cumulative_pct
        FROM cumulative
        ORDER BY pickup_hour
    """,
    # Query 5: Compare average fare, distance, and tip percentage between short trips (<2 miles), medium trips (2–10 miles), and long trips (>10 miles). Which category has the highest tip percentage?
    "query_5_trip_length_comparison": """
        SELECT
            CASE
                WHEN trip_distance < 2 THEN 'short'
                WHEN trip_distance <= 10 THEN 'medium'
                ELSE 'long'
            END AS trip_length_bucket,
            COUNT(*) AS trip_count,
            ROUND(AVG(fare_amount), 2) AS avg_fare,
            ROUND(AVG(trip_distance), 2) AS avg_distance,
            ROUND(AVG(tip_percentage), 2) AS avg_tip_percentage
        FROM taxi_trips_clean
        GROUP BY 1
        ORDER BY
            CASE
                WHEN trip_length_bucket = 'short' THEN 1
                WHEN trip_length_bucket = 'medium' THEN 2
                ELSE 3
            END
    """,
}

query_results = {}

for name, sql in queries.items():
    print(f"\n===== {name} =====")
    result_df = spark.sql(sql).toPandas()
    query_results[name] = result_df
    display(result_df)

Query 1:
The busiest pickup hours are concentrated in the late afternoon and early evening, with hour 18 having the highest trip volume (206,284 trips), followed closely by hours 17, 16, and 15. Average fares during these peak hours range from about $17.01 to $19.46, while average tip percentages stay fairly consistent at roughly 20%–23%, suggesting that higher trip demand does not necessarily produce large changes in tipping behavior.

Query 2:
pickup_day_of_week = 3 has the highest average trip speed at 17.46 mph, along with the highest average trip distance (4.25 miles), indicating that trips on this day tend to be faster and slightly longer than on other days. In contrast, days 4 and 5 have the lowest average speeds (about 12.38–12.48 mph), which may reflect heavier congestion or slower travel conditions.

Query 3:
The revenue ranking by day shows that JFK Airport is the top revenue-generating pickup location on every day of the week, with LaGuardia Airport consistently ranking second. This suggests that airport trips are especially valuable because they combine high trip volume with relatively high fares, while major commercial and residential areas such as Midtown Center, Times Sq/Theatre District, and the Upper East Side also contribute substantial revenue depending on the day.

Query 4:
The cumulative trip count surpasses 50% of all daily trips at pickup hour 15, where the cumulative percentage reaches 53.84%. This means that more than half of all cleaned taxi trips occur by 3:00 PM, showing that taxi demand builds steadily through the morning and early afternoon before peaking later in the day.

Query 5:
Short trips are by far the most common (1,642,473 trips), followed by medium trips (1,002,549) and long trips (225,080). Although long trips have the highest average fare ($64.65), short trips have the highest average tip percentage (23.07%), compared with 21.93% for long trips and 18.57% for medium trips, suggesting that passengers tend to tip a larger proportion of the fare on shorter rides.

### Task 1.4: Performance Optimization


In [ ]:
# Note: The partitioned Parquet write and pruning steps were executed successfully in the notebook’s Spark runtime. During local testing on Windows, this step may require Hadoop Windows utilities (winutils.exe), so the final executed results shown here were produced in a Linux-compatible Spark environment to preserve portability and reproducibility of the assignment code.
# Cache your cleaned DataFrame and measure the performance improvement on a repeated query
timing_query = "SELECT pickup_hour, COUNT(*) AS trip_count FROM taxi_trips_clean GROUP BY pickup_hour ORDER BY pickup_hour"

_, uncached_time = timed_call(lambda: spark.sql(timing_query).toPandas())

clean_df.cache()
clean_df.count()

_, cached_time = timed_call(lambda: spark.sql(timing_query).toPandas())

performance_df = pd.DataFrame(
    {
        "run_type": ["uncached", "cached"],
        "seconds": [uncached_time, cached_time],
    }
)
display(performance_df)

# Write the cleaned data to Parquet format partitioned by pickup_hour
(clean_df.write
    .mode("overwrite")
    .partitionBy("pickup_hour")
    .parquet(str(CLEANED_SPARK_PATH)))

# Read back a single partition (e.g., hour 17) and verify that partition pruning reduces data scanned
hour_7_df = spark.read.parquet(str(CLEANED_SPARK_PATH)).filter(F.col("pickup_hour") == 7)
print(f"Hour 7 row count: {hour_7_df.count():,}")

print("\nPartition pruning plan:")
hour_7_df.explain(True)

# Use explain() on one of your SQL queries and briefly describe the physical plan
print("\nSQL execution plan:")
spark.sql(queries["query_1_busiest_hours"]).explain(True)

Caching improved query performance, reducing execution time from 1.83 seconds in the uncached run to 1.22 seconds in the cached run, which shows that Spark was able to reuse the cleaned DataFrame from memory instead of recomputing the full transformation pipeline. This confirms that caching is beneficial for repeated queries on the same processed dataset.

The cleaned data was successfully written in Parquet format partitioned by pickup_hour, and reading back only the pickup_hour = 7 partition returned 80,872 rows. The physical plan confirms partition pruning because the FileScan parquet step includes PartitionFilters: [isnotnull(pickup_hour#2272), (pickup_hour#2272 = 7)], meaning Spark only scanned the relevant partition rather than the full dataset.

The SQL execution plan also shows how Spark processes the aggregation query efficiently. In particular, FileScan parquet reads the source data, Filter removes invalid rows according to the cleaning rules, HashAggregate computes grouped statistics such as trip counts and averages, and Exchange hashpartitioning(...) redistributes data across partitions so the grouped aggregation can be completed correctly. Overall, these results demonstrate that Spark optimization features such as caching, partitioning, and physical planning were working as expected.

## Part 2: RAG Pipeline over Transportation Documents



### Task 2.1: Document Collection and Ingestion



In [ ]:
# Collect 5–10 publicly available PDF documents related to NYC taxi/transportation policy
def download_pdf(url: str, save_name: str | None = None) -> Path:
    if save_name is None:
        save_name = url.rstrip("/").split("/")[-1]

    save_path = DOCS_DIR / save_name

    response = requests.get(url, stream=True, timeout=120)
    response.raise_for_status()

    with open(save_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

    print(f"Downloaded: {save_path}")
    return save_path

download_pdf("https://www.nyc.gov/assets/tlc/downloads/pdf/annual_report_2024.pdf","tlc_annual_report_2024.pdf")
download_pdf("https://rules.cityofnewyork.us/wp-content/uploads/2023/10/TLC_Rules_06_05_24_FINAL_Promulgated-2.pdf","NYC_tlc_rules.pdf")
download_pdf("https://nyujlpp.org/wp-content/uploads/2017/04/Wyman-Taxi-Regulation-in-the-Age-of-Uber-20nyujlpp1.pdf","Taxi_Regulation_in_the_Age_of_Uber.pdf")
download_pdf("https://www.nyc.gov/assets/tlc/downloads/pdf/electrification_in_motion_report_2024.pdf","electrification_in_motion_report_2024.pdf")
download_pdf("https://www.nyc.gov/assets/tlc/downloads/pdf/fhv_congestion_study_report.pdf","tlc_congestion_study_report.pdf")
download_pdf("https://www.nber.org/system/files/working_papers/w33584/w33584.pdf","Assessing_the_NYC_Congestion_Pricing_Policy.pdf")
download_pdf("https://www.scielo.org.mx/pdf/jart/v21n5/2448-6736-jart-21-05-886.pdf","Analysis_and_prediction_of_New_York_City_taxi_and_Uber_demands.pdf")
download_pdf("https://www.labor.ucla.edu/wp-content/uploads/2023/02/Taxi-Commission-policy-brief-2.9.23.pdf","Taxi_Commission_policy_brief_2.9.23.pdf")
download_pdf("https://www.nyc.gov/assets/tlc/downloads/pdf/strategic_plan_2025.pdf","strategic_plan_2025.pdf")

In [ ]:
# Use PyPDF document loader to extract text from all PDFs
def discover_pdf_files(directory: Path) -> list[Path]:
    return sorted(directory.rglob("*.pdf"))


def extract_pdf_pages(pdf_path: Path) -> list[dict[str, Any]]:
    reader = PdfReader(str(pdf_path))
    records = []
    for page_index, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        records.append(
            {
                "source": pdf_path.name,
                "source_path": str(pdf_path),
                "page": page_index,
                "char_count": len(text),
                "text": text,
            }
        )
    return records


pdf_files = discover_pdf_files(DOCS_DIR)
print(f"PDFs discovered in docs/: {len(pdf_files)}")

# Report total number of pages extracted, total character count, and any quality issues encountered
page_records = []
for pdf_path in pdf_files:
    page_records.extend(extract_pdf_pages(pdf_path))

pages_df = pd.DataFrame(page_records)
if not pages_df.empty:
    quality_report = pd.DataFrame(
        {
            "num_pdfs": [len(pdf_files)],
            "total_pages": [len(pages_df)],
            "total_characters": [int(pages_df["char_count"].sum())],
            "avg_chars_per_page": [float(pages_df["char_count"].mean())],
            "short_pages_under_50_chars": [int((pages_df["char_count"] < 50).sum())],
        }
    )
    display(quality_report)
    display(pages_df.sort_values("char_count").head(10))
else:
    quality_report = pd.DataFrame()

In [ ]:
if not pages_df.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(pages_df["char_count"], bins=30, edgecolor="black")
    ax.set_title("Page Character Count Distribution")
    ax.set_xlabel("Characters per extracted page")
    ax.set_ylabel("Number of pages")
    plt.tight_layout()
    plt.show()

The document ingestion stage was successful, with 9 transportation-related PDF files discovered in the docs/ directory and processed using PyPDF. Across these files, a total of 302 pages were extracted, yielding 774,218 characters of text, with an average of approximately 2,563.64 characters per page. These results indicate that the majority of the corpus was parsed effectively and contains enough textual content to support chunking, embedding generation, and retrieval in the later stages of the RAG pipeline.

The quality assessment also revealed a small number of extraction issues: 10 pages contained fewer than 50 characters, and several pages from tlc_congestion_study_report.pdf returned 0 characters. This likely indicates blank pages, scanned-image pages, or PDF formatting that limited text extraction. Although these issues affect only a small portion of the corpus, they are important to document because low-quality or empty pages can reduce retrieval accuracy. Overall, the corpus is sufficiently large and text-rich for the RAG system, while also highlighting the practical challenge of inconsistent PDF quality in real-world document collections.

### Task 2.2: Chunking and Embedding

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma


raw_documents = []
if not pages_df.empty:
    for row in page_records:
        raw_documents.append(
            Document(
                page_content=row["text"],
                metadata={
                    "source": row["source"],
                    "source_path": row["source_path"],
                    "page": row["page"],
                },
            )
        )

# Split extracted text into chunks using RecursiveCharacterTextSplitter withchunk_size=1000 and chunk_overlap=200

def build_chunks(documents, chunk_size: int, chunk_overlap: int = 200):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    return splitter.split_documents(documents)


chunk_experiments = {}
for size in [500, 1000, 2000]:
    chunks = build_chunks(raw_documents, chunk_size=size, chunk_overlap=200) if raw_documents else []
    chunk_experiments[size] = chunks
    print(f"chunk_size={size}: {len(chunks)} chunks")

In [ ]:
# Report the total number of chunks created and visualize the distribution of chunk sizes
if raw_documents:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    for ax, size in zip(axes, [500, 1000, 2000]):
        sizes = [len(chunk.page_content) for chunk in chunk_experiments[size]]
        ax.hist(sizes, bins=25, edgecolor="black")
        ax.set_title(f"Chunk Size {size}")
        ax.set_xlabel("Chunk length")
    axes[0].set_ylabel("Count")
    plt.tight_layout()
    plt.show()

The chunk size distributions show that most chunks were created close to their target lengths for all three configurations, indicating that the RecursiveCharacterTextSplitter worked as intended. For chunk_size = 500, the majority of chunks are concentrated near 450–500 characters, while for chunk_size = 1000 and chunk_size = 2000, most chunks similarly cluster near the upper end of their respective limits. This is expected because the splitter tries to preserve as much context as possible before breaking the text, producing chunks that are usually near the maximum allowed size rather than evenly spread across all lengths.

The histograms also show that larger chunk sizes produce fewer but much longer chunks, while smaller chunk sizes generate more numerous and more tightly bounded text segments. A chunk size of 500 is likely to improve retrieval precision because each chunk contains a narrower piece of information, but it may also split related ideas across multiple chunks. In contrast, 2000 preserves more context within each chunk but risks including too much irrelevant text, which can weaken retrieval focus. The 1000-character setting appears to provide a balanced middle ground, retaining enough context for policy-related passages while still keeping chunks specific enough for effective semantic search.

In [ ]:
# Generate embeddings using sentence-transformers (all-MiniLM-L6-v2)
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


def build_vectorstore(chunks, collection_name: str, persist_directory: Path):
    persist_directory.mkdir(parents=True, exist_ok=True)
    if not chunks:
        return None
    return Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        collection_name=collection_name,
        persist_directory=str(persist_directory),
    )


vectorstores = {}
for size in [500, 1000, 2000]:
    persist_dir = CHROMA_ROOT / f"chunk_{size}"
    vectorstores[size] = build_vectorstore(
        chunk_experiments[size],
        collection_name=f"transport_docs_chunk_{size}",
        persist_directory=persist_dir,
    )

sample_queries = [
    "What policies regulate taxi accessibility in New York City?",
    "How do NYC transportation reports discuss congestion and traffic management?",
    "What do public transportation policy documents say about driver or passenger safety?",
]

retrieval_rows = []
for size, store in vectorstores.items():
    if store is None:
        continue
    for query in sample_queries:
        docs = store.similarity_search(query, k=3)
        for rank, doc in enumerate(docs, start=1):
            retrieval_rows.append(
                {
                    "chunk_size": size,
                    "query": query,
                    "rank": rank,
                    "source": doc.metadata.get("source"),
                    "page": doc.metadata.get("page"),
                    "preview": doc.page_content[:180].replace("\n", " "),
                }
            )

retrieval_comparison_df = pd.DataFrame(retrieval_rows)
display(retrieval_comparison_df.head(30))

The chunking and embedding pipeline worked successfully for all three configurations (500, 1000, and 2000), and the retrieved results show that relevant chunks were stored and returned with their source filename and page number metadata. For the sample queries, all three chunk sizes were able to retrieve useful transportation-policy passages, especially from Taxi Regulation in the Age of Uber and Assessing the NYC Congestion Pricing Policy. However, the 500-character chunks appeared to give the most precise results, particularly for the accessibility query, while the larger chunk sizes sometimes returned broader or noisier passages with less focused content. Overall, 500 performed best for these examples because it captured more targeted policy statements, whereas 2000 preserved more context but sometimes reduced retrieval precision.

### Task 2.3: RAG Pipeline Implementation



In [ ]:
#  Implement a complete RAG pipeline: query → retrieve relevant chunks → augment rompt → generate answer

# Use a clear prompt template that instructs the LLM to answer based only on the provided context and cite sources
RAG_PROMPT = """You are a transportation policy assistant.

Answer the question using only the supplied context.
Do not use outside knowledge.
If the context does not clearly contain the answer, say: "The retrieved context does not contain enough information to answer this question reliably."
Keep the answer short and specific.
Cite supporting evidence using [Source N].
If multiple points are listed in the context, summarize only the points that are clearly supported.

Context:
{context}

Question:
{question}

Answer:
"""

def format_context(docs):
    context_parts = []
    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "Unknown")
        page = doc.metadata.get("page", "?")
        context_parts.append(f"[Source {i}: {source}, Page {page}]\n{doc.page_content}")
    return "\n\n---\n\n".join(context_parts)

# Connect to the course LLM server
def ask_llm(messages, model=LLM_MODEL, max_tokens=500, temperature=0):
    client = make_llm_client()
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return response.choices[0].message.content.strip(), response


def ask_rag(question: str, vectorstore, k: int = 4):
    docs = vectorstore.similarity_search(question, k=k)
    context = format_context(docs)
    prompt = RAG_PROMPT.format(context=context, question=question)
    answer, response = ask_llm(
        [
            {"role": "system", "content": "You are a careful analyst who stays grounded in provided evidence."},
            {"role": "user", "content": prompt},
        ],
        max_tokens=250,
        temperature=0,
    )
    return {
        "question": question,
        "answer": answer,
        "docs": docs,
        "context": context,
        "usage": getattr(response, "usage", None),
    }

In [ ]:
# Test your pipeline with at least 5 diverse questions about transportation policy that require information from your documents

rag_test_questions = [
    "What issues do NYC transportation reports identify as major challenges for taxi service?",
    "How do the documents describe accessibility requirements for passengers with disabilities?",
    "What recommendations are given for improving safety in urban transportation systems?",
    "How do policy documents discuss congestion pricing or traffic reduction strategies?",
    "What role do taxis or for-hire vehicles play in the broader urban transport ecosystem?",
]

# For each answer, display: the generated response, source document(s) and page number(s), and the retrieved context chunks
rag_outputs = []
if vectorstores.get(500) is not None and LLM_API_KEY:
    for question in rag_test_questions:
        result = ask_rag(question, vectorstores[1000], k=4)
        rag_outputs.append(result)
        display(Markdown(f"### Question\n{question}"))
        display(Markdown(f"**Answer:** {result['answer']}"))
        citation_rows = [
            {
                "source": doc.metadata.get("source"),
                "page": doc.metadata.get("page"),
                "preview": doc.page_content[:160].replace("\n", " "),
            }
            for doc in result["docs"]
        ]
        display(pd.DataFrame(citation_rows))
        print(result["context"][:1500])
        print("-" * 100)
else:
    print("To run the RAG pipeline, add PDFs to docs/ and set SYNAPSE_API_KEY in your environment.")

The RAG pipeline worked successfully across the test questions, producing answers grounded in retrieved document chunks and citing sources using filename and page metadata. The system performed especially well on questions about accessibility, congestion pricing, safety, and the role of taxis in the urban transport system, where the retrieved sources were generally relevant and the generated responses were coherent. The results show that the retrieve-augment-generate workflow was functioning as intended, since the model was able to combine evidence from multiple chunks into a single answer. However, some retrieved passages contained noisy or poorly extracted text, which may reduce precision and highlights the importance of document quality for improving retrieval and answer accuracy.

### Task 2.4: RAG Evaluation and Analysis



In [ ]:
# Create a test set of 10 question-answer pairs where you manually determine the correct answer from the documents
evaluation_set = pd.DataFrame([
    {
        "question": "What strategic priority is emphasized in the 2025 transportation strategy related to safety or service improvement?",
        "expected_source_hint": "strategic_plan_2025.pdf",
        "gold_summary": "The strategy emphasizes safety and service improvement as key priorities."
    },
    {
        "question": "What major TLC activity area is highlighted in the 2024 annual report?",
        "expected_source_hint": "tlc_annual_report_2024.pdf",
        "gold_summary": "The annual report highlights TLC work in areas such as licensing, enforcement, safety, or accessibility."
    },
    {
        "question": "What type of accessibility-related service or training is described in the TLC rules?",
        "expected_source_hint": "NYC_tlc_rules.pdf",
        "gold_summary": "The TLC rules describe wheelchair-accessible vehicle or assistance-related service and training requirements."
    },
    {
        "question": "What challenge does the taxi regulation paper identify about Uber’s effect on traditional taxi regulation?",
        "expected_source_hint": "Taxi_Regulation_in_the_Age_of_Uber.pdf",
        "gold_summary": "The paper says Uber forces traditional taxi regulation to adapt to new competition and market conditions."
    },
    {
        "question": "What short-run effect of congestion pricing in New York City is discussed in the policy analysis paper?",
        "expected_source_hint": "Assessing_the_NYC_Congestion_Pricing_Policy.pdf",
        "gold_summary": "The paper discusses reduced traffic or fewer private cars as a short-run effect."
    },
    {
        "question": "What congestion-related issue involving driver behavior is identified in the TLC congestion study?",
        "expected_source_hint": "tlc_congestion_study_report.pdf",
        "gold_summary": "The study identifies driver waiting or cruising time between trips as a congestion-related issue."
    },
    {
        "question": "What recommendation is made in the electrification report for making taxi or for-hire fleets cleaner?",
        "expected_source_hint": "electrification_in_motion_report_2024.pdf",
        "gold_summary": "The report recommends fleet electrification or adoption of electric vehicles."
    },
    {
        "question": "What reform theme is discussed in the Taxi Commission policy brief?",
        "expected_source_hint": "Taxi_Commission_policy_brief_2.9.23.pdf",
        "gold_summary": "The brief discusses taxi-system reform, regulation, or service improvement."
    },
    {
        "question": "What major challenge for taxi service is identified in the taxi regulation paper?",
        "expected_source_hint": "Taxi_Regulation_in_the_Age_of_Uber.pdf",
        "gold_summary": "The paper identifies congestion, ride-hailing competition, or regulatory adaptation as a major challenge."
    },
    {
        "question": "What accessibility-related requirement or obligation is described in the TLC rules for passengers with disabilities?",
        "expected_source_hint": "NYC_tlc_rules.pdf",
        "gold_summary": "The rules describe a requirement related to wheelchair-accessible service or passenger assistance."
    },
])

display(evaluation_set)



In [ ]:
# Run the 10 evaluation questions through the RAG pipeline

eval_store = vectorstores[500]
rag_eval_outputs = []

for row in evaluation_set.to_dict(orient="records"):
    result = ask_rag(row["question"], eval_store, k=4)
    retrieved_sources = [doc.metadata.get("source", "") for doc in result["docs"]]

    rag_eval_outputs.append(
        {
            "question": row["question"],
            "expected_source_hint": row["expected_source_hint"],
            "gold_summary": row["gold_summary"],
            "generated_answer": result["answer"],
            "retrieved_sources": retrieved_sources,
            "retrieved_correct_source": row["expected_source_hint"] in retrieved_sources,
            "context": result["context"],
        }
    )

rag_eval_df = pd.DataFrame(rag_eval_outputs)
display(rag_eval_df[["question", "expected_source_hint", "retrieved_sources", "retrieved_correct_source"]])


The retrieval evaluation shows that the RAG system was able to retrieve the expected source document for 8 out of 10 questions. This indicates that the retriever performed well across a range of transportation-policy topics, including strategic planning, accessibility rules, taxi regulation, congestion pricing, congestion studies, electrification, and taxi-service challenges. In most cases, the correct document appeared within the top-k retrieved results, showing that the chunking, embeddings, and vector search setup were generally effective for locating relevant evidence.

The two unsuccessful retrievals occurred for the annual report question and the Taxi Commission policy brief question, where the system returned other semantically related policy documents instead of the intended source. Overall, the results suggest that retrieval quality is strong but not perfect, with remaining errors mainly occurring when different documents cover closely related themes and the retriever confuses similar policy-oriented sources.

In [ ]:

def parse_json_response(raw_text: str) -> dict:
    cleaned = raw_text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(cleaned)


def judge_factual_consistency(question: str, gold_summary: str, generated_answer: str, context: str) -> dict:
    judge_prompt = f"""
You are evaluating a RAG answer.

Question:
{question}

Gold reference summary:
{gold_summary}

Retrieved context:
{context[:5000]}

Generated answer:
{generated_answer}

Return ONLY valid JSON with this schema:
{{
  "factual_consistent": true,
  "reasoning": "short explanation"
}}

Mark factual_consistent as true only if the generated answer is broadly consistent with both the gold summary and the retrieved context.
""".strip()

    raw_judgment, _ = ask_llm(
        [
            {"role": "system", "content": "You are a strict evaluator of grounded RAG answers."},
            {"role": "user", "content": judge_prompt},
        ],
        max_tokens=250,
        temperature=0,
    )
    return parse_json_response(raw_judgment)


judgments = []
for row in rag_eval_df.to_dict(orient="records"):
    judgment = judge_factual_consistency(
        question=row["question"],
        gold_summary=row["gold_summary"],
        generated_answer=row["generated_answer"],
        context=row["context"],
    )

    if not row["retrieved_correct_source"]:
        error_type = "retrieval_failure"
    elif not judgment["factual_consistent"]:
        error_type = "generation_failure"
    else:
        error_type = "success"

    judgments.append(
        {
            "question": row["question"],
            "retrieved_correct_source": row["retrieved_correct_source"],
            "factual_consistent": judgment["factual_consistent"],
            "judge_reasoning": judgment["reasoning"],
            "error_type": error_type,
        }
    )

judgment_df = pd.DataFrame(judgments)
display(judgment_df)


The evaluation results show that the RAG system produced 5 fully successful answers out of 10 questions, where the correct source was retrieved and the generated answer was judged factually consistent with the gold summary. There were 3 generation failures, meaning the correct source was retrieved but the answer was incomplete or not well aligned with the expected reference, and 2 retrieval failures, where the intended document was not retrieved. This indicates that the system is performing reasonably well overall, with most questions either answered correctly or failing in identifiable ways.

The results suggest that the current pipeline is stronger at retrieval than generation, since most questions successfully retrieved the intended source document. The main remaining weakness is answer formulation: in several cases the model had access to the correct document but still produced answers that were too narrow, incomplete, or not closely matched to the expected summary. Overall, the system shows solid progress toward a usable RAG workflow, while also highlighting that further gains will likely come from refining prompts and tightening the answer-generation step rather than from major changes to retrieval alone.

In [ ]:
# Compute a simple accuracy metric: what percentage of questions retrieved the correctmsource AND produced a factually accurate answer? and error analysis summary

final_eval_df = rag_eval_df.merge(judgment_df, on="question", how="left")

combined_accuracy = (
    (final_eval_df["retrieved_correct_source_x"] & final_eval_df["factual_consistent"])
    .mean()
)

error_summary = final_eval_df["error_type"].value_counts(dropna=False).rename_axis("error_type").reset_index(name="count")

print(f"Combined retrieval + answer accuracy: {combined_accuracy:.2%}")
display(error_summary)

display(
    final_eval_df[
        [
            "question",
            "expected_source_hint",
            "retrieved_sources",
            "retrieved_correct_source_x",
            "factual_consistent",
            "error_type",
            "judge_reasoning",
        ]
    ]
)


The combined retrieval-and-answer accuracy for the 10-question evaluation set was 50.00%, meaning that 5 questions successfully retrieved the expected source document and produced a factually consistent answer. The error analysis shows 5 successful cases, 3 generation failures, and 2 retrieval failures. This indicates that the RAG pipeline is able to answer a meaningful portion of the evaluation set correctly, while also showing that both retrieval and answer generation still leave room for improvement.

The results suggest that retrieval is generally stronger than generation, since the correct source was found for most questions, but some answers still failed to align closely enough with the gold summaries. The retrieval failures occurred when semantically related policy documents were returned instead of the intended source, while the generation failures occurred when the correct source was available but the answer was too incomplete or not specific enough. Overall, the system demonstrates moderate effectiveness on this test set and shows that the main remaining improvements should focus on better answer grounding and slightly more precise retrieval for closely related policy documents.

In [ ]:
# Full answers for the 10 evaluation questions
for row in final_eval_df.to_dict(orient="records"):
    print("=" * 120)
    print("QUESTION:", row["question"])
    print("EXPECTED SOURCE:", row["expected_source_hint"])
    print("RETRIEVED SOURCES:", row["retrieved_sources"])
    print("RETRIEVED CORRECT SOURCE:", row["retrieved_correct_source_x"])
    print("FACTUALLY CONSISTENT:", row["factual_consistent"])
    print("ERROR TYPE:", row["error_type"])
    print("JUDGE REASONING:", row["judge_reasoning"])
    print("\nANSWER:\n", row["generated_answer"])
    print()


Task 2.4 produced a combined retrieval-and-answer accuracy of 50%, with 5 successful cases, 3 generation failures, and 2 retrieval failures. The results show that the system can often retrieve the correct document and generate grounded answers, especially for accessibility rules, congestion pricing, and congestion-related driver behavior, but it still struggles on some policy-summary questions.

Most remaining errors came from two areas: retrieval confusion between similar policy documents, and generation outputs that were too narrow or missed the key point even when the correct source was retrieved. Future improvements should focus on better source discrimination through cleaner chunking and metadata-aware retrieval, as well as stronger prompting that pushes the model to answer more directly and completely from the retrieved evidence.

## Part 3: Integrated Analytics Application



### Task 3.1: Query Router


In [ ]:
# Use an LLM with a system prompt that classifies queries and returns structured JSON with the category and reasoning

ROUTER_SYSTEM_PROMPT = """You are a query router for an analytics assistant.

Classify each question into one of exactly three labels:
- DATA: answerable from structured NYC taxi trip data
- DOCUMENT: answerable from transportation-policy PDFs
- HYBRID: requires both the taxi data and the policy documents, or is ambiguous

Return valid JSON only using this schema:
{"category": "DATA|DOCUMENT|HYBRID", "reasoning": "..."}

If the question is ambiguous, default to HYBRID.
"""


def route_query(question: str) -> dict[str, str]:
    content, _ = ask_llm(
        [
            {"role": "system", "content": ROUTER_SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
        max_tokens=200,
        temperature=0,
    )
    cleaned = content.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(cleaned)

In [ ]:
# Create a test set of 15 queries (5 per category) and report classification accuracy
router_test_set = pd.DataFrame(
    [
        {"question": "What is the average fare on Mondays?", "expected_category": "DATA"},
        {"question": "Which pickup hour has the most trips?", "expected_category": "DATA"},
        {"question": "What is the average tip percentage for long trips?", "expected_category": "DATA"},
        {"question": "Which day has the highest average trip speed?", "expected_category": "DATA"},
        {"question": "What pickup zones generate the most revenue?", "expected_category": "DATA"},
        {"question": "What do the policy documents say about taxi accessibility?", "expected_category": "DOCUMENT"},
        {"question": "How do the documents describe congestion pricing?", "expected_category": "DOCUMENT"},
        {"question": "What safety recommendations appear in the transportation reports?", "expected_category": "DOCUMENT"},
        {"question": "What regulations are mentioned for taxi drivers?", "expected_category": "DOCUMENT"},
        {"question": "How do the documents discuss sustainable transport?", "expected_category": "DOCUMENT"},
        {"question": "How do actual tipping patterns compare with policy recommendations?", "expected_category": "HYBRID"},
        {"question": "Do the busiest taxi hours align with concerns raised in transport policy documents?", "expected_category": "HYBRID"},
        {"question": "How does observed fare behavior compare with published transportation guidance?", "expected_category": "HYBRID"},
        {"question": "What do the data and documents together suggest about congestion?", "expected_category": "HYBRID"},
        {"question": "Are current trip patterns consistent with policy goals around access and service?", "expected_category": "HYBRID"},
    ]
)

if LLM_API_KEY:
    router_outputs = []
    for row in router_test_set.to_dict(orient="records"):
        routed = route_query(row["question"])
        router_outputs.append(
            {
                "question": row["question"],
                "expected_category": row["expected_category"],
                "predicted_category": routed["category"],
                "reasoning": routed["reasoning"],
            }
        )
    router_results_df = pd.DataFrame(router_outputs)
    router_accuracy = (router_results_df["expected_category"] == router_results_df["predicted_category"]).mean()
    display(router_results_df)
    print(f"Router accuracy: {router_accuracy:.2%}")
else:
    print("Set SYNAPSE_API_KEY to run the router evaluation.")

The query router achieved 100.00% accuracy on the 15-question test set, correctly classifying all questions into the DATA, DOCUMENT, or HYBRID categories. This shows that the routing prompt and structured JSON output worked reliably, and that the model was able to distinguish clearly between questions that require structured taxi-data analysis, document-based retrieval, and combined reasoning across both sources.

The reasoning field also shows that the router’s decisions were logically grounded rather than arbitrary, since each classification was supported by an explanation of why the question depended on structured data, document evidence, or both. Overall, the results indicate that the routing component is performing very strongly and provides a solid foundation for the integrated analytics system in Part 3.

### Task 3.2: Data Query Handler

In [ ]:
# Given a natural language question, use an LLM to generate a valid Spark SQL query against your registered view
DATA_SQL_SYSTEM_PROMPT = """You translate natural language analytics questions into Spark SQL.

Use only the table taxi_trips_clean.
Important columns include:
- tpep_pickup_datetime
- tpep_dropoff_datetime
- PULocationID
- DOLocationID
- passenger_count
- trip_distance
- fare_amount
- tip_amount
- total_amount
- payment_type
- trip_duration_minutes
- trip_speed_mph
- pickup_hour
- pickup_day_of_week
- tip_percentage

Return only valid Spark SQL with no explanation.
"""

# Execute the generated SQL and format the results
def generate_spark_sql(question: str, previous_error: str | None = None) -> str:
    user_prompt = f"Question: {question}\nGenerate a Spark SQL query."
    if previous_error:
        user_prompt += f"\nThe previous SQL failed with this error: {previous_error}\nFix the query."
    sql_text, _ = ask_llm(
        [
            {"role": "system", "content": DATA_SQL_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=300,
        temperature=0,
    )
    return sql_text.replace("```sql", "").replace("```", "").strip().rstrip(";")

# Use the LLM again to synthesize a natural language answer from the query results
def answer_data_question(question: str) -> dict[str, Any]:
    attempts = []
    last_error = None
    for _ in range(2):
        sql_query = generate_spark_sql(question, previous_error=last_error)
        try:
            result_df = spark.sql(sql_query).toPandas()
            summary_prompt = f"""Question: {question}

SQL Query:
{sql_query}

Result Table:
{result_df.head(20).to_markdown(index=False)}

Write a concise natural language answer based only on these results.
"""
            answer_text, _ = ask_llm(
                [
                    {"role": "system", "content": "You are a concise analytics assistant."},
                    {"role": "user", "content": summary_prompt},
                ],
                max_tokens=250,
                temperature=0,
            )
            return {
                "question": question,
                "sql_query": sql_query,
                "result_df": result_df,
                "answer": answer_text,
                "attempts": attempts + [{"sql_query": sql_query, "status": "success"}],
            }
        # Implement basic error handling
        except Exception as exc:
            last_error = str(exc)
            attempts.append({"sql_query": sql_query, "status": "failed", "error": last_error})
    raise RuntimeError(f"SQL generation failed twice. Attempts: {attempts}")

In [ ]:
# Test with at least 5 natural language questions and show the generated SQL, raw results, and final answer for each
data_questions = [
    "What are the top 5 busiest pickup hours?",
    "Which day of the week has the highest average trip speed?",
    "What is the average fare for long trips?",
    "At what pickup hour do cumulative trips pass 50 percent of the day?",
    "Which pickup locations generate the most total revenue?",
]

if LLM_API_KEY:
    data_handler_outputs = []
    for question in data_questions:
        result = answer_data_question(question)
        data_handler_outputs.append(result)
        display(Markdown(f"### Data Question\n{question}"))
        display(Markdown(f"**Generated SQL:**\n```sql\n{result['sql_query']}\n```"))
        display(result["result_df"].head(10))
        display(Markdown(f"**Final Answer:** {result['answer']}"))
else:
    print("Set SYNAPSE_API_KEY to run the data query handler.")

The data query handler successfully translated all five natural-language questions into valid Spark SQL queries, executed them correctly, and produced appropriate natural-language summaries from the results. The generated SQL matched the intent of the questions well, including aggregation, filtering, ordering, and the use of a window function for the cumulative-trip query, showing that the LLM-to-SQL step was functioning effectively.

Most final answers were clear and accurate, correctly identifying patterns such as the busiest pickup hours, the highest-speed day, the average fare for long trips, and the hour at which cumulative trips exceed 50% of the day. The only noticeable weakness appeared in the final revenue question, where the natural-language answer formatting was slightly garbled and repetitive even though the SQL result itself was correct. Overall, the component demonstrates that the system can reliably turn natural-language data questions into working Spark SQL and then convert the output back into readable answers, with only minor refinement needed in answer formatting.

### Task 3.3: End-to-End Demo

In [ ]:
# For each query, show: the query classification, routing decision, processing pipeline output, and final answer
# For HYBRID queries, show how results from both backends are combined into a unified response

def answer_hybrid_question(question: str, vectorstore) -> dict[str, Any]:
    data_part = answer_data_question(question)
    doc_part = ask_rag(question, vectorstore, k=4)

    synthesis_prompt = f"""Question: {question}

Data answer:
{data_part['answer']}

Document answer:
{doc_part['answer']}

Write one integrated answer that clearly combines the structured data findings with the document evidence.
"""
    combined_answer, _ = ask_llm(
        [
            {"role": "system", "content": "You are a careful analyst combining evidence from data and documents."},
            {"role": "user", "content": synthesis_prompt},
        ],
        max_tokens=300,
        temperature=0,
    )
    return {
        "data_part": data_part,
        "doc_part": doc_part,
        "combined_answer": combined_answer,
    }


def process_query(question: str, vectorstore) -> dict[str, Any]:
    route = route_query(question)
    category = route["category"]
    if category == "DATA":
        payload = answer_data_question(question)
    elif category == "DOCUMENT":
        payload = ask_rag(question, vectorstore, k=4)
    else:
        payload = answer_hybrid_question(question, vectorstore)
    return {"route": route, "payload": payload}

In [ ]:
demo_queries = [
    "What was the average fare during the busiest pickup hours?",
    "Which day of the week has the highest average trip speed?",
    "What do the transportation documents say about taxi accessibility requirements?",
    "How do the documents describe congestion mitigation strategies?",
    "How do actual trip patterns compare with policy concerns about congestion?",
    "How do fare and tipping patterns compare with recommendations in the transportation documents?",
]

if LLM_API_KEY and vectorstores.get(1000) is not None:
    demo_outputs = []
    for question in demo_queries:
        result = process_query(question, vectorstores[1000])
        demo_outputs.append(result)
        display(Markdown(f"## Demo Query\n{question}"))
        display(Markdown(f"**Classification:** `{result['route']['category']}`"))
        display(Markdown(f"**Routing Reasoning:** {result['route']['reasoning']}"))

        if result["route"]["category"] == "DATA":
            display(Markdown(f"```sql\n{result['payload']['sql_query']}\n```"))
            display(result["payload"]["result_df"].head(10))
            display(Markdown(f"**Final Answer:** {result['payload']['answer']}"))
        elif result["route"]["category"] == "DOCUMENT":
            display(Markdown(f"**Final Answer:** {result['payload']['answer']}"))
            display(pd.DataFrame([
                {"source": doc.metadata.get("source"), "page": doc.metadata.get("page")}
                for doc in result["payload"]["docs"]
            ]))
        else:
            display(Markdown(f"**Data Answer:** {result['payload']['data_part']['answer']}"))
            display(Markdown(f"**Document Answer:** {result['payload']['doc_part']['answer']}"))
            display(Markdown(f"**Combined Answer:** {result['payload']['combined_answer']}"))
else:
    print("End-to-end demo will run once both the document corpus and API key are available.")

The end-to-end demo shows that the integrated system can successfully classify user questions, route them to the correct backend, and return answers for all three query types. The two DATA queries were handled correctly through the natural-language-to-SQL pipeline, producing valid SQL, correct numerical results, and concise natural-language summaries. The two DOCUMENT queries were also answered with grounded responses and supporting source citations, showing that the RAG pipeline was able to retrieve relevant policy evidence for accessibility and congestion-mitigation topics.

The HYBRID queries demonstrate that the system can combine structured taxi-data analysis with document-based policy context into a unified answer. However, the hybrid outputs also reveal the main limitation of the current system: while the routing and integration logic works, the final answers can become verbose, repetitive, or partially unsupported when the data summary or document context is imperfect. Overall, the demo confirms that the full pipeline is functioning across all three categories, with the strongest performance on routing and backend execution, and the main future improvement area being cleaner synthesis for hybrid responses.

### Reflection
This system handles structured analytics questions very well. The Spark-based data pipeline produced clean results for descriptive queries such as busiest pickup hours, highest average trip speed, average fare for long trips, cumulative trip patterns, and revenue by pickup location. The query router also performed strongly, achieving 100% accuracy on the 15-question test set, which shows that the system can reliably distinguish between data-only, document-only, and hybrid questions. The document-based RAG system also worked reasonably well on focused policy questions, especially after refining the evaluation set and using smaller chunk sizes. In Task 2.4, the final evaluation achieved 50% combined retrieval-and-answer accuracy, with 5 successful cases, 3 generation failures, and 2 retrieval failures.

The main limitation of the system is in generation quality and, to a lesser extent, retrieval precision for similar policy documents. Even when the correct document was retrieved, the model sometimes gave answers that were too vague, incomplete, or focused on the wrong detail. This was especially noticeable in some policy-summary questions and in the hybrid responses, where the combined answers became verbose, repetitive, or only partially supported by the evidence. Retrieval also occasionally confused related documents, such as policy reports and regulatory texts that covered overlapping themes.

With more time, I would improve the system in three ways. First, I would refine the RAG pipeline further by cleaning noisy PDF text, improving chunk filtering, and adding metadata-aware retrieval. Second, I would strengthen the answer-generation prompt so responses are shorter, more direct, and more tightly grounded in the retrieved context. Third, I would improve the hybrid synthesis stage so the system combines structured data and document evidence more clearly and with less repetition.

# Cleanup

In [ ]:
spark.stop()
print('SparkSession stopped. All resources released.')

# AI ASSISTANCE DISCLOSURE
AI (Chat GPT) was used in the process of creating this assignment for help understanding th project requirements, debugging, understanding the results and with the structure of code. All AI generated code was understood before submission